# agentest quickstart

Judge a recorded good agent and a buggy copy. CPU only. No account, API key, GPU or model. Nothing here downloads a model.

This file lives in the git repository. GitHub renders it. A separate docs site is not required.

Colab opens this file from the default branch with no GitHub login:

https://colab.research.google.com/github/srpadmanabhuni/agentest/blob/main/examples/colab_quickstart.ipynb

Or, from a checkout:

```bash
python examples/colab_quickstart.py
```

The runnable steps are in `examples/colab_quickstart.py`. In Colab the next cell clones the default branch, installs PyYAML if it is missing, and runs that script. On a checkout it uses the files already on disk.

In Colab, if the `datasets` package is already installed, the script loads one catalogue row from `srinivaspadmanabhuni/agentic-ai-test-checklist-110`, config `all`, split `train` (the live default). That is one cached read of the catalogue file. If `datasets` is not installed, the demo still finishes and the script prints a skip. This notebook installs PyYAML only. An explicit load (`--load-dataset` or `AGENTEST_COLAB_LOAD_DATASET=1`) needs `datasets` and stops if it is missing. `datasets` is not an agentest dependency.

Dataset card: https://huggingface.co/datasets/srinivaspadmanabhuni/agentic-ai-test-checklist-110

Walkthroughs of single checks, a plain OpenTelemetry trace and a JUnit CI workflow are already in the repo. The script prints those paths. It does not add a second copy.

Contact: srinivas AT testAIng.com


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def find_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents, here / "agentest"]
    for p in candidates:
        if (p / "examples" / "colab_quickstart.py").is_file() and (p / "src" / "agentest" / "__init__.py").is_file():
            return p
    dest = here / "agentest"
    if not dest.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/srpadmanabhuni/agentest.git", str(dest)])
    if not (dest / "examples" / "colab_quickstart.py").is_file():
        raise SystemExit(
            "This checkout has the demo agents but not examples/colab_quickstart.py yet. "
            "That file is on the branch that contains this notebook. "
            "After merge, cloning the default branch is enough."
        )
    return dest

ROOT = find_root()
os.chdir(ROOT)
if importlib.util.find_spec("yaml") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml"])
print("repo", ROOT)


## Run the demo

`refill_bot` is the clean agent. `refill_bot_buggy` is the same agent with seeded defects. The script prints the CLI summary, then a table of the failing checks only. `n/a` is not a pass.


In [ ]:
import runpy

try:
    runpy.run_path(str(ROOT / "examples" / "colab_quickstart.py"), run_name="__main__")
except SystemExit as exc:
    if exc.code not in (0, None):
        raise


## What you just saw

The clean agent has zero failing checks in this demo. The buggy agent fails checks that the walkthroughs already explain (verify-before-acting, instructions in tool output, an unearned success claim, personal data left unmasked).

Next, still in the repository:

- `docs/walkthroughs/README.md`
- `examples/otel_plain/emit_trace.py` (one OTLP/JSON trace, standard library only)
- `docs/developers/CI.md` (`agentest std run --format junit --out agentest.xml --strict`)
- `docs/GETTING-STARTED.md`
The script also prints one temporary repair: it records `verify_identity` before the dependent calls on a copy of the buggy log, then reruns. `T2-001` and `T2-005` pass on that copy. Other failures remain. That is not a pass rate. The shipped traces are not modified. `agentest-ai` is not on PyPI; this notebook installs PyYAML only.
